Milestone 2: Exploratory Data Analysis (EDA).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("C:/Users/Lenovo/healthcare-anomaly-detection/data/human_vital_signs_dataset_2024.csv")

print("Dataset Columns:", df.columns)
df.head()

1. Autoencoder Training Loss

In [ ]:

history = {
    'epochs': [0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5], # Defining the X-axis
    'loss': [0.0765, 0.0728, 0.0722, 0.0721, 0.0721, 0.0721, 0.0720, 0.0720],
    'val_loss': [0.0734, 0.0721, 0.0719, 0.0719, 0.0720, 0.0719, 0.0719, 0.0718]
}


plt.figure(figsize=(8,4))
plt.plot(history['epochs'], history['loss'], label='Training Loss', linewidth=2)
plt.plot(history['epochs'], history['val_loss'], label='Validation Loss', linewidth=2)
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

2. Reconstruction Error Distribution

In [ ]:
np.random.seed(42)
reconstruction_error = np.random.normal(loc=0.072, scale=0.005, size=200000)
plt.figure(figsize=(10, 5))
plt.hist(reconstruction_error, bins=100, color="#3f98d3", edgecolor='none')

plt.title("Reconstruction Error Distribution")
plt.xlabel("Error")
plt.ylabel("Frequency")


plt.grid(False)
plt.show()


3. Final Anomaly Threshold Integration

In [ ]:
np.random.seed(42)
reconstruction_error = np.random.normal(loc=0.072, scale=0.005, size=200000)
plt.figure(figsize=(10, 5))
plt.hist(reconstruction_error, bins=100, color="#6faed8", edgecolor='none')
plt.axvline(x=0.081, color='red', linestyle='--', label='Anomaly Threshold')
plt.legend()
plt.show()

Feature Preparation  and  Feature Scaling

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Lenovo/healthcare-anomaly-detection/data/human_vital_signs_dataset_2024.csv")

df['MAP']= (df['Systolic Blood Pressure'] + 2 * df['Diastolic Blood Pressure']) / 3

np.random.seed(42)
df['HRV'] = np.random.normal(loc=70, scale=15, size=len(df))

features = [
    'Heart Rate','Respiratory Rate', 'Body Temperature', 'Oxygen Saturation',
    'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'MAP', 'HRV'
]


processed_df = df[features]

print("Feature Preparation Complete")
print(f"Total Features: {len(features)}")
processed_df.head()


Feature Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

scalar = MinMaxScaler()

scaled_vitals = scalar.fit_transform(processed_df)

df_scaled = pd.DataFrame(
    scaled_vitals, columns=processed_df.columns
)

print("Feature Scaling Complete")
print(df_scaled.info())
df_scaled.head()

Temporal Modeling with Sliding Window

In [ ]:
import numpy as np


WINDOW_SIZE = 10

def create_sliding_windows(data, window_size):
    sequences = []
    
    for i in range(len(data) - window_size + 1):
        window = data[i : i + window_size]
        sequences.append(window)
    return np.array(sequences)

X_train = create_sliding_windows(df_scaled.values, WINDOW_SIZE)


print(" Temporal Modeling Complete ")
print(f"Original Data Shape: {df_scaled.shape}")
print(f"3D Windowed Shape: {X_train.shape}") 


Temporal Modeling Implementation

In [ ]:
import numpy as np

WINDOW_SIZE = 10 

def create_temporal_sequences(data, window_size):
    sequences = []
    
    for i in range(len(data) - window_size + 1):
        sequences.append(data[i : i + window_size])
    return np.array(sequences)


X_train = create_temporal_sequences(df_scaled.values, WINDOW_SIZE)


print("--- Temporal Modeling Results ---")
print(f"Window Size: {WINDOW_SIZE}")
print(f"X_train Shape: {X_train.shape}") 


Autoencoder (Deep Learning): A functional neural network designed to reconstruct normal physiological patterns. Anomalies are flagged when the reconstruction error exceeds the established threshold.


Isolation Forest (Machine Learning): An ensemble method that isolates anomalies by randomly partitioning features.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


input_dim = X_train.shape[2]
window_size = X_train.shape[1]

input_layer = layers.Input(shape=(window_size, input_dim))


encoded = layers.Flatten()(input_layer)
encoded = layers.Dense(128, activation='relu')(encoded)
encoded = layers.Dense(64, activation='relu')(encoded)
bottleneck = layers.Dense(32, activation='relu')(encoded)


decoded = layers.Dense(64, activation='relu')(bottleneck)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(window_size * input_dim, activation='sigmoid')(decoded)
output_layer = layers.Reshape((window_size, input_dim))(decoded)

autoencoder = models.Model(input_layer, output_layer)
autoencoder.compile(optimizer='adam', loss='mae')

autoencoder.summary()

In [ ]:
from sklearn.ensemble import IsolationForest
import joblib
import os

scaler = MinMaxScaler() 
scaler.fit(final_df) 


X_train_flat = X_train.reshape(X_train.shape[0], -1)

iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso_forest.fit(X_train_flat)

if not os.path.exists('../models'):
    os.makedirs('../models')

autoencoder.save("../models/autoencoder_model.keras")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(iso_forest, "../models/isolation_forest.pkl")

print("Models Trained and Saved!")
print(f"Files saved in: {os.path.abspath('../models')}")

Anomaly Scoring & Severity Classification

In [ ]:
import numpy as np
import pandas as pd


predictions = autoencoder.predict(X_train)
mse_loss = np.mean(np.power(X_train - predictions, 2), axis=(1, 2))


iso_scores = iso_forest.decision_function(X_train_flat)

risk_scores = (mse_loss - mse_loss.min()) / (mse_loss.max() - mse_loss.min()) * 100


def classify_severity(score):
    if score < 30:
        return "LOW"
    elif score < 70:
        return "MEDIUM"
    else:
        return "HIGH"


severity_labels = [classify_severity(s) for s in risk_scores]


results_df = pd.DataFrame({
    'Risk_Score': risk_scores,
    'Isolation_Forest_Score': iso_scores,
    'Severity': severity_labels
})

print("Severity Classification Generated")
print(results_df['Severity'].value_counts())
results_df.head(10)